In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import speech_recognition as sr
from pydub import AudioSegment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from wordcloud import WordCloud
from collections import Counter
import re

c:\Users\Jonny Villareal\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
#Datos de filtros
fecha_i = '20260301'
fecha_f = '20260331'

mes = 'marzo'

In [3]:
#imprortar paradas

paradas = pd.read_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Paraderos_Zonales_del_SITP.csv')

paradas.head(3)

,X,Y,objectid,cenefa,zona_sitp,nombre,via,direccion_bandera,localidad,longitud,latitud,consecutivo_zona,tipo_m_s,consola,panel,audio,zonas_nuevas,globalid,shape
0,1.001502e+06,1.010205e+06,1,001A00,00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481,001,M,AC 100 - KR 54 (001A00),AC 100 - KR 54,Avenida Calle 100 Carrera 54,C,{1C0DBC4E-15BC-4BBE-BC16-5628077DBE2E},NaN
1,1.003505e+06,1.009719e+06,2,001A01,01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091,001,M,AC 100 - KR 13 (001A01),AC 100 - KR 13,Avenida Calle 100 Carrera 13,B,{60A22A44-AD56-4DF4-A3E6-6830B9FB0792},NaN
2,1.001238e+06,1.018098e+06,3,001A02,02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867,001,S,AV. Boyacá - AC 170 (001A02),AV. Boyacá - AC 170,Avenida Boyacá Avenida Calle 170,C,{97155274-E4D5-45A4-891F-2DCA19FF10D9},NaN


In [4]:
# Ruta de la carpeta que contiene los archivos de actividad de bus zonal
ruta_carpeta = 'Z:/01 base_datos/08 varados FMS'

# Fechas de inicio y fin para el filtro
fecha_inicio = f'{fecha_i}'
fecha_fin = f'{fecha_f}'

# Lista para almacenar los DataFrames
dataframes = []

# Recorrer todos los archivos en la carpeta
for nombre_archivo in os.listdir(ruta_carpeta):
    if nombre_archivo.endswith('_varados.csv'):
        # Extraer la fecha del nombre del archivo
        fecha_archivo = nombre_archivo[:8]  
        
        # Convertir la fecha a un formato adecuado para comparación
        fecha_archivo_dt = pd.to_datetime(fecha_archivo, format='%Y%m%d', errors='coerce')

        # Verificar si la fecha está dentro del rango deseado
        if fecha_inicio <= fecha_archivo <= fecha_fin:
            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)
            # Leer el archivo CSV omitiendo la primera fila vacía
            df = pd.read_csv(ruta_archivo, encoding='latin', low_memory=False)
            dataframes.append(df)

# Verificar si se encontraron DataFrames
if dataframes:
    # Consolidar todos los DataFrames en uno solo
    notas = pd.concat(dataframes, ignore_index=True)

    # # Guardar el DataFrame consolidado en un nuevo archivo
    # desg_troncal.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Indicadores/Indicadores_python/desgl_troncal_ago24.csv', index=False)
else:
    print("No se encontraron archivos para consolidar.")
    
notas.head(3)

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/03/2026 3:10,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10292,577,0,0,,802826,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización de Desvío / ID 93132 / 01-03-2026...,WILLIAM MUNAR HUMBERTO GONZALEZ,WILLIAM MUNAR HUMBERTO GONZALEZ
1,1/03/2026 3:10,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10292,577,0,0,,802827,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 90406 / 01-03-2026 / ...,WILLIAM MUNAR HUMBERTO GONZALEZ,WILLIAM MUNAR HUMBERTO GONZALEZ
2,1/03/2026 3:14,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10325,576,0,0,,802828,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 87938 / 01-03-2026 / ...,WILLIAM MUNAR HUMBERTO GONZALEZ,WILLIAM MUNAR HUMBERTO GONZALEZ


In [6]:
#Filtrar por tipo nota igual a 'INTERRUPCION DE SERVICIO ZONAL', 'NOVEDADES SIRCI', 'RUTAS EN SEGUIMIENTO ESPECIAL'
notas = notas[notas["Tipo Nota"].isin([
    "INTERRUPCION DE SERVICIO ZONAL"
])]

notas.head(3)

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/03/2026 3:10,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10292,577,0,0,,802826,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización de Desvío / ID 93132 / 01-03-2026...,WILLIAM MUNAR HUMBERTO GONZALEZ,WILLIAM MUNAR HUMBERTO GONZALEZ
1,1/03/2026 3:10,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10292,577,0,0,,802827,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 90406 / 01-03-2026 / ...,WILLIAM MUNAR HUMBERTO GONZALEZ,WILLIAM MUNAR HUMBERTO GONZALEZ
2,1/03/2026 3:14,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10325,576,0,0,,802828,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 87938 / 01-03-2026 / ...,WILLIAM MUNAR HUMBERTO GONZALEZ,WILLIAM MUNAR HUMBERTO GONZALEZ


In [7]:
def parse_observacion(texto):
    if pd.isna(texto):
        return {}

    pares = re.findall(r'([^:/]+):\s*([^/]+)', texto)
    return {k.strip(): v.strip() for k, v in pares}

df_obs = notas["Observaciones"].apply(parse_observacion).apply(pd.Series)

df_obs.head(3)


,03,Paradas,Paraderos Omitidos,Reportado por,ID,Desde,Sentido,Descripción de desvío,Reportado por Regulador de CCZ,05,...,02,R Reportado por,2026 10,DIRECCION,SENTIDO,AFECTACION,GESTION,2026 11,Siendo las 12,Siendo las 14
0,00,Entre 099D09_Carvajal - Hasta 510B09_Br. Alque...,3,RCC Willian Humberto Munar González,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00,NaN,NaN,NaN,90406,052A08: Hasta: 114A08,Occidente - Oriente,toman av. américas hasta carrera 76k retoman e...,Willian Humberto Munar González,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00,NaN,NaN,NaN,87938,330A08: Hasta: 541A08,Oriente - Occidente,Tomar por la Calle 53 Sur al Occidente Hasta l...,Willian Humberto Munar González,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
def parse_observacion(texto):
    if pd.isna(texto):
        return {}

    pares = re.findall(r'([^:/]+):\s*([^/]+)', texto)
    return {k.strip(): v.strip() for k, v in pares}


# Parsear observaciones
df_obs = notas["Observaciones"].apply(parse_observacion).apply(pd.Series)

# Agregar columnas base
df_obs = pd.concat(
    [notas[["Fecha","Id de línea","Nombre Línea","ID Nota"]], df_obs],
    axis=1
)

df_obs.head(3)

,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,Desde,...,02,R Reportado por,2026 10,DIRECCION,SENTIDO,AFECTACION,GESTION,2026 11,Siendo las 12,Siendo las 14
0,1/03/2026 3:10,10292,577,802826,00,Entre 099D09_Carvajal - Hasta 510B09_Br. Alque...,3,RCC Willian Humberto Munar González,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1/03/2026 3:10,10292,577,802827,00,NaN,NaN,NaN,90406,052A08: Hasta: 114A08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1/03/2026 3:14,10325,576,802828,00,NaN,NaN,NaN,87938,330A08: Hasta: 541A08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df_obs = df_obs.sort_values(by="ID", key=lambda x: x.isna())

df_obs.head(3)

,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,Desde,...,02,R Reportado por,2026 10,DIRECCION,SENTIDO,AFECTACION,GESTION,2026 11,Siendo las 12,Siendo las 14
2493,15/03/2026 1:29,10196,E25,850699,00,NaN,NaN,NaN,95735,050A00: Hasta: 227A00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3706,22/03/2026 1:30,10342,KB309,874377,00,NaN,NaN,NaN,82281,441A01: Hasta: 064A01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3705,22/03/2026 1:29,10342,KB309,874376,00,NaN,NaN,NaN,107762,355B00: Hasta: 913A00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_obs.insert(
    0,
    "tipo_evento",
    np.where(df_obs["ID"].notna() & (df_obs["ID"] != ""), "Desvios", "Novedades de ruta")
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,02,R Reportado por,2026 10,DIRECCION,SENTIDO,AFECTACION,GESTION,2026 11,Siendo las 12,Siendo las 14
2493,Desvios,15/03/2026 1:29,10196,E25,850699,00,NaN,NaN,NaN,95735,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3706,Desvios,22/03/2026 1:30,10342,KB309,874377,00,NaN,NaN,NaN,82281,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3705,Desvios,22/03/2026 1:29,10342,KB309,874376,00,NaN,NaN,NaN,107762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df_obs.columns = df_obs.columns.str.strip()

In [12]:
#Limpiar texto
mask = df_obs["tipo_evento"] == "Desvios"

extraidos = (
    df_obs.loc[mask, "Desde"]
    .astype(str)
    .str.findall(r"\d+[A-Z]\d+")
)

df_obs.loc[mask, "Parada_ini"] = extraidos.str[0]
df_obs.loc[mask, "Parada_fin"] = extraidos.str[1]

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,2026 10,DIRECCION,SENTIDO,AFECTACION,GESTION,2026 11,Siendo las 12,Siendo las 14,Parada_ini,Parada_fin
2493,Desvios,15/03/2026 1:29,10196,E25,850699,00,NaN,NaN,NaN,95735,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,050A00,227A00
3706,Desvios,22/03/2026 1:30,10342,KB309,874377,00,NaN,NaN,NaN,82281,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,441A01,064A01
3705,Desvios,22/03/2026 1:29,10342,KB309,874376,00,NaN,NaN,NaN,107762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,355B00,913A00


In [ ]:
print(df_obs["Paradas"].str.contains("Entre", case=False, na=False).sum())

965

In [14]:
# Limpieza base
df_obs["Paradas"] = (
    df_obs["Paradas"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
)

# Extracción robusta
extraido = df_obs["Paradas"].str.extract(
    r'Entre\s*(.*?)\s*-\s*Hasta\s*(.*)',
    flags=re.IGNORECASE
)

# Aplicar solo si es Novedades de ruta
mask = df_obs["tipo_evento"].str.contains("Novedades", case=False, na=False)

df_obs.loc[mask, "parada_ini_afectada"] = extraido[0]
df_obs.loc[mask, "parada_fin_afectada"] = extraido[1]

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,SENTIDO,AFECTACION,GESTION,2026 11,Siendo las 12,Siendo las 14,Parada_ini,Parada_fin,parada_ini_afectada,parada_fin_afectada
2493,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,NaN,NaN,NaN,NaN,NaN,NaN,050A00,227A00,NaN,NaN
3706,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,NaN,NaN,NaN,NaN,NaN,NaN,441A01,064A01,NaN,NaN
3705,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,NaN,NaN,NaN,NaN,NaN,NaN,355B00,913A00,NaN,NaN


In [15]:
# Regex para extraer código de parada
patron = r'(\d{3}[A-Za-z]\d{2})'

df_obs["parada_ini_afec"] = df_obs["parada_ini_afectada"].str.extract(patron)
df_obs["parada_fin_afec"] = df_obs["parada_fin_afectada"].str.extract(patron)

df_obs["parada_ini_afec"] = df_obs["parada_ini_afec"].str.upper()
df_obs["parada_fin_afec"] = df_obs["parada_fin_afec"].str.upper()

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,GESTION,2026 11,Siendo las 12,Siendo las 14,Parada_ini,Parada_fin,parada_ini_afectada,parada_fin_afectada,parada_ini_afec,parada_fin_afec
2493,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,NaN,NaN,NaN,NaN,050A00,227A00,NaN,NaN,NaN,NaN
3706,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,NaN,NaN,NaN,NaN,441A01,064A01,NaN,NaN,NaN,NaN
3705,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,NaN,NaN,NaN,NaN,355B00,913A00,NaN,NaN,NaN,NaN


In [16]:
#cruzar datos de paradas con paradas de la bitácora

maestro_paradas = paradas[[
    "cenefa",
    "nombre",
    "via",
    "direccion_bandera",
    "localidad",
    "longitud",
    "latitud"
]]

maestro_paradas.head()

,cenefa,nombre,via,direccion_bandera,localidad,longitud,latitud
0,001A00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481
1,001A01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091
2,001A02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867
3,001A03,Br. Julio Flórez,AC 100,AC 100 - KR 66A,Suba,-74.071439,4.689504
4,001A04,Avenida Calle 80,AK 68,AK 68 - CL 79D,Barrios Unidos,-74.080392,4.682626


In [17]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_ini"),
    left_on="Parada_ini",
    right_on="cenefa_ini",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,parada_fin_afectada,parada_ini_afec,parada_fin_afec,cenefa_ini,nombre_ini,via_ini,direccion_bandera_ini,localidad_ini,longitud_ini,latitud_ini
0,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,NaN,NaN,NaN,050A00,Br. Colombia,Av. Chile,AC 72 - KR 22,Barrios Unidos,-74.066179,4.662240
1,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,NaN,NaN,NaN,441A01,Br. Tibabita,KR 8C,KR 8C - CL 185B,Usaquén,-74.029942,4.763045
2,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,NaN,NaN,NaN,355B00,Br. Concepción Norte,KR 17,KR 17 - CL 67,Barrios Unidos,-74.065826,4.656064


In [18]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_fin"),
    left_on="Parada_fin",
    right_on="cenefa_fin",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,localidad_ini,longitud_ini,latitud_ini,cenefa_fin,nombre_fin,via_fin,direccion_bandera_fin,localidad_fin,longitud_fin,latitud_fin
0,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,Barrios Unidos,-74.066179,4.662240,227A00,Clínica de Traumatología y Ortopedia,AK 11,AK 11 - CL 70A,Chapinero,-74.059659,4.656000
1,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,Usaquén,-74.029942,4.763045,064A01,Gimnasio Las Palmas,AK 7,AK 7 - CL 186,Usaquén,-74.027455,4.763086
2,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,Barrios Unidos,-74.065826,4.656064,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_ini_afec"),
    left_on="parada_ini_afec",
    right_on="cenefa_ini_afec",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,localidad_fin,longitud_fin,latitud_fin,cenefa_ini_afec,nombre_ini_afec,via_ini_afec,direccion_bandera_ini_afec,localidad_ini_afec,longitud_ini_afec,latitud_ini_afec
0,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,Chapinero,-74.059659,4.656000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,Usaquén,-74.027455,4.763086,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_obs = df_obs.merge(
    maestro_paradas.add_suffix("_fin_afec"),
    left_on="parada_fin_afec",
    right_on="cenefa_fin_afec",
    how="left"
)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,localidad_ini_afec,longitud_ini_afec,latitud_ini_afec,cenefa_fin_afec,nombre_fin_afec,via_fin_afec,direccion_bandera_fin_afec,localidad_fin_afec,longitud_fin_afec,latitud_fin_afec
0,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
columnas_cero = [
    "longitud_fin_afec",
    "latitud_fin_afec",
    "longitud_ini_afec",
    "latitud_ini_afec",
    "latitud_ini",
    "longitud_ini",
    "latitud_fin",
    "longitud_fin"
]

df_obs[columnas_cero] = df_obs[columnas_cero].fillna(0)

df_obs.head(3)

,tipo_evento,Fecha,Id de línea,Nombre Línea,ID Nota,03,Paradas,Paraderos Omitidos,Reportado por,ID,...,localidad_ini_afec,longitud_ini_afec,latitud_ini_afec,cenefa_fin_afec,nombre_fin_afec,via_fin_afec,direccion_bandera_fin_afec,localidad_fin_afec,longitud_fin_afec,latitud_fin_afec
0,Desvios,15/03/2026 1:29,10196,E25,850699,00,nan,NaN,NaN,95735,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,Desvios,22/03/2026 1:30,10342,KB309,874377,00,nan,NaN,NaN,82281,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,Desvios,22/03/2026 1:29,10342,KB309,874376,00,nan,NaN,NaN,107762,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0


In [22]:
print(df_obs['latitud_ini_afec'].unique())

[0.         4.60178015 4.60749454 4.6460606  4.71689461 4.64953457
 4.65823957 4.60884778 4.62261171 4.65157026 4.555117   4.66113505
 4.6042917  4.57597306 4.65746279 4.56096701 4.5594871  4.58221416
 4.672921   4.65064725 4.66998312 4.59345452 4.66944024 4.65111749
 4.67052663 4.56443772 4.67333148 4.61608037 4.5693859  4.64366426
 4.760222   4.760348   4.58789498 4.675746   4.63051206 4.687194
 4.66353923 4.69686489 4.478735   4.63060007 4.61736456 4.595634
 4.59498552 4.66725267 4.67441202 4.55099149 4.703389   4.56393612
 4.67372494 4.67535218 4.66263352 4.68400257 4.69033645 4.57547435
 4.58534386 4.68322252 4.6262458  4.67936135 4.67594882 4.68320848
 4.66736444 4.68186234 4.675564   4.68374383 4.69962016 4.71264116
 4.70522754 4.70590762 4.6742381  4.65635708 4.59400231 4.59186761
 4.600383   4.57160399 4.68338161 4.680646   4.57148757 4.68209102
 4.5735905  4.5899546  4.54379563 4.54953528 4.62231624 4.61881354
 4.61605125 4.61585259 4.71019252 4.55876892 4.68050166 4.680281
 

In [23]:
df_obs.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Rutas_puntos_críticos/factores_rutas_{mes}.csv', sep=';', index=False)